# Introduction

In [18]:
import numpy as np

# Problem 1: Generating Random Boolean Functions
The Deutsch–Jozsa algorithm is designed to work with functions that accept a fixed number of Boolean inputs and return a single Boolean output. Each function is guaranteed to be either constant (always returns False or always returns True) or balanced (returns True for exactly half of the possible input combinations). Write a Python function random_constant_balanced that returns a randomly chosen function from the set of constant or balanced functions taking four Boolean arguments as inputs.

---
### Background Context

The **Deutsch-Jozsa algorithm** is one of the earliest quantum algorithms that demonstrates a clear advantage over classical computation. Developed in 1992, it proves that quantum computers can solve certain problems exponentially faster than classical computers by exploiting quantum superposition and interference.

**The Problem:**  
Given a black-box function (called an "oracle") that takes $n$ Boolean inputs and returns a single Boolean output, determine whether the function is:
- **Constant:** Always returns the same output (all 0s or all 1s) regardless of input
- **Balanced:** Returns 0 for exactly half of the inputs and 1 for the other half

We're guaranteed the function is one of these two types (this is called a "promise problem").

**Classical vs. Quantum:**  
Classically, in the worst case, you'd need to check $2^{n-1} + 1$ inputs to be certain (e.g., for 4 bits, potentially 9 of the 16 inputs). The quantum algorithm solves this in exactly **1 query**, regardless of $n$.

---
### Approach and Reasoning

#### `random_constant_balanced(rng=None)`
This function generates either a constant or balanced Boolean function by first randomly choosing which type to create. For constant functions, I simply select a Boolean value and return a closure that always outputs that value regardless of input. 

For balanced functions, I generate all 16 possible input combinations using `itertools.product`, then use `random.sample` to select exactly 8 inputs that will return `True`—this guarantees the balanced property (exactly half true) rather than relying on probabilistic coin flips which could give 7 or 9 true outputs. The returned function checks if its current input tuple is in the pre-selected list of "true inputs." 

I included an optional `rng` parameter to allow seeded random generators for reproducible testing while defaulting to the global `random` module for convenience.

---

#### `classify_constant_or_balanced(f)`
This classifier exhaustively evaluates the function on all 16 possible input combinations and counts how many return `True`:
- **Count = 0 or 16:** Constant (all outputs identical)
- **Count = 8:** Balanced (half true, half false)  
- **Otherwise:** Returns "neither" to catch invalid functions

This brute-force approach mirrors the classical computational challenge—we must check every input to be certain—which contrasts with the quantum Deutsch-Jozsa algorithm that determines the property in a single query. I store all outputs in a list before counting to keep the logic clear and enable potential future extensions like displaying the full truth table.

---

#### `format_truth_table(f)`
This helper function generates a readable truth table by iterating through all input combinations in 0/1 format (for conventional Boolean logic display), converting them to Boolean values to call the function, then converting the result back to 0/1 for output. 

I chose explicit conversion with `bool()` and an `if-else` statement rather than shortcuts to avoid type confusion and make the data flow crystal clear. The function returns a list of formatted strings rather than printing directly, following good separation of concerns—the caller decides how to display the results.

---

#### Verification Strategy

**Assert statements:**  
These three assertions verify that my classifier correctly identifies known constant and balanced functions. By testing with hand-crafted functions whose properties I can verify independently, I ensure the classification logic is sound before testing it on randomly generated functions.

**Random trials loop:**  
This loop generates 5 random functions and asserts each is classified as either constant or balanced (never "neither"). This statistical test verifies that my generator never produces invalid functions—no matter what random choices are made, the output always satisfies the problem constraints. I chose 5 trials as a balance between runtime and confidence.

**Seeded RNG test:**  
This test uses a seeded random number generator (`Random(123)`) to create a reproducible function, then verifies its true-count is valid (0, 8, or 16). The seed ensures this test produces identical results every run, making it suitable for automated testing and debugging. By checking the raw count rather than just the classification string, I verify the mathematical property directly.

**Demonstration loop:**  
This loop generates two random functions and displays their complete truth tables with classifications. The output lets me visually verify that balanced functions truly have 8 ones and 8 zeros, that the patterns appear random (different between runs), and that the implementation works end-to-end.

---
### Implementation

In [19]:
import random
from itertools import product

def random_constant_balanced(rng=None):
    #if no random generator is provided, use the standard one
    if rng is None:
        rng = random
    
    #decide whether to create a constant or balanced function
    choice = rng.choice(["constant", "balanced"])

    if choice == "constant":
        #pick a single output value (True or False)
        value = rng.choice([False, True])
        
        #define a function that always returns that value, ignoring inputs
        def constant_function(a, b, c, d):
            return value
            
        return constant_function

    if choice == "balanced":
        #generate all 16 possible combinations of 4 inputs
        all_possible_inputs = list(product([False, True], repeat=4))
        
        #randomly pick exactly 8 of them to return true
        inputs_that_return_true = rng.sample(all_possible_inputs, k=8)
        
        #define a function that checks if the input is in our chosen list
        def balanced_function(a, b, c, d):
            current_input = (a, b, c, d)
            if current_input in inputs_that_return_true:
                return True
            else:
                return False
                
        return balanced_function

In [20]:
def classify_constant_or_balanced(f):
    #create a list of all 16 inputs to test the function
    inputs = list(product([False, True], repeat=4))
    
    #evaluate the function for every input and store the results
    outputs = []
    for inp in inputs:
        result = f(*inp)
        outputs.append(result)
    
    #count how many times the function returned true
    true_count = sum(outputs)

    #constant if all outputs are the same (all 0 or all 16 are true)
    if true_count == 0 or true_count == 16:
        return "constant"
        
    #balanced if exactly half (8) are true
    if true_count == 8:
        return "balanced"
        
    return "neither"

In [21]:
#test cases
def test_constant_true(a, b, c, d):
    #always returns true (constant)
    return True


def test_constant_false(a, b, c, d):
    #always returns false (constant)
    return False


def test_balanced_parity(a, b, c, d):
    #even parity gives a balanced function over 4 bits
    return (a + b + c + d) % 2 == 0


#run tests
assert classify_constant_or_balanced(test_constant_true) == "constant"
print("test_constant_true: passed")
assert classify_constant_or_balanced(test_constant_false) == "constant"
print("test_constant_false: passed")
assert classify_constant_or_balanced(test_balanced_parity) == "balanced"
print("test_balanced_parity: passed")

test_constant_true: passed
test_constant_false: passed
test_balanced_parity: passed


In [22]:
def format_truth_table(f):
    lines = []
    for inp in product([0, 1], repeat=4):
        #convert 0/1 integers to boolean False/True for the function
        a = bool(inp[0])
        b = bool(inp[1])
        c = bool(inp[2])
        d = bool(inp[3])
        
        #get the output from the function
        result = f(a, b, c, d)
        
        #convert the result back to 0 or 1 for printing
        if result == True:
            out = 1
        else:
            out = 0
            
        lines.append(f"{inp} -> {out}")
    return lines

In [23]:
#random trials should always be constant or balanced by construction
for _ in range(5):
    f = random_constant_balanced()
    assert classify_constant_or_balanced(f) in {"constant", "balanced"}

#seeded rng makes the test deterministic
#use a specific seed so we get the same function every time we run this
seeded_rng = random.Random(123)
f_seeded = random_constant_balanced(seeded_rng)

#check if the function is valid using our helper
result = classify_constant_or_balanced(f_seeded)

if result == "constant" or result == "balanced":
    print(f"seeded test passed: generated a {result} function.")
else:
    print("seeded test failed.")

#results and demonstration: truth tables
for i in range(1, 3):
    f = random_constant_balanced()
    label = classify_constant_or_balanced(f)
    print("Truth Table:")
    print(label)
    print(f"Try {i}:")
    for line in format_truth_table(f):
        print(line)
    print()

seeded test passed: generated a constant function.
Truth Table:
balanced
Try 1:
(0, 0, 0, 0) -> 0
(0, 0, 0, 1) -> 1
(0, 0, 1, 0) -> 0
(0, 0, 1, 1) -> 0
(0, 1, 0, 0) -> 0
(0, 1, 0, 1) -> 0
(0, 1, 1, 0) -> 1
(0, 1, 1, 1) -> 1
(1, 0, 0, 0) -> 1
(1, 0, 0, 1) -> 0
(1, 0, 1, 0) -> 1
(1, 0, 1, 1) -> 1
(1, 1, 0, 0) -> 1
(1, 1, 0, 1) -> 0
(1, 1, 1, 0) -> 0
(1, 1, 1, 1) -> 1

Truth Table:
constant
Try 2:
(0, 0, 0, 0) -> 0
(0, 0, 0, 1) -> 0
(0, 0, 1, 0) -> 0
(0, 0, 1, 1) -> 0
(0, 1, 0, 0) -> 0
(0, 1, 0, 1) -> 0
(0, 1, 1, 0) -> 0
(0, 1, 1, 1) -> 0
(1, 0, 0, 0) -> 0
(1, 0, 0, 1) -> 0
(1, 0, 1, 0) -> 0
(1, 0, 1, 1) -> 0
(1, 1, 0, 0) -> 0
(1, 1, 0, 1) -> 0
(1, 1, 1, 0) -> 0
(1, 1, 1, 1) -> 0



### References - make look pretty later

https://quantum.country/qcvc

# Problem 2: Classical Testing for Function Type
Deutsch's algorithm is designed to demonstrate a potential advantage of quantum computing over classical computation. To understand this advantage, we must first understand the classical cost of solving the underlying problem. Write a Python function determine_constant_balanced that takes as input a function f, as defined in Problem 1. The function should analyze f and return the string "constant" or "balanced" depending on whether the function is constant or balanced. Write a brief note on the efficiency of your solution. What is the maximum number of times you need to call f to be 100% certain whether it is constant or balanced?

In [24]:
def determine_constant_balanced(f):
    all_inputs = list(product([False, True], repeat=4))
    
    #get the first output to use as reference
    first_output = f(*all_inputs[0])
    
    #check the remaining 15 inputs to see if any differ from the first
    for inp in all_inputs[1:]:  #start from index 1 (skip the first one we already checked)
        output = f(*inp)
        
        #if we find a different output, it can't be constant
        if output != first_output:
            #since we're promised it's either constant or balanced,
            #and it's not constant, it must be balanced!
            return "balanced"
    
    #if we got here, all outputs matched the first one
    #that means the function is constant
    return "constant"

def determine_constant_balanced_with_counter(f):
    all_inputs = list(product([False, True], repeat=4))
    
    call_count = 0  #track how many times we call f
    
    #get the first output to use as reference
    first_output = f(*all_inputs[0])
    call_count += 1  #we just made our first call
    
    #check the remaining 15 inputs to see if any differ from the first
    for inp in all_inputs[1:]:  #start from index 1
        output = f(*inp)
        call_count += 1  #increment each time we call f
        
        #if we find a different output, we can stop early
        if output != first_output:
            #it's not constant, so it must be balanced
            return "balanced", call_count
    
    #if we got here, all outputs matched the first one
    return "constant", call_count

In [ ]:

#optimal version of determine_constant_balanced that stops as early as possible
#a balanced function has exactly 8 True and 8 False outputs
#so if we ever see 9 outputs that are all the same, it must be constant
#and if we see any output that differs from the first, it must be balanced
#this means we never need to check more than 9 inputs

def determine_constant_balanced_optimal(f):
    all_inputs = list(product([False, True], repeat=4))
    call_count = 0

    #check the first input and remember what it returned
    first_output = f(*all_inputs[0])
    call_count += 1

    for inp in all_inputs[1:]:
        output = f(*inp)
        call_count += 1

        #if this output is different from the first, it cant be constant
        #since we know its either constant or balanced, it must be balanced
        if output != first_output:
            return "balanced", call_count

        #if we have seen 9 identical outputs, it cant be balanced
        #balanced functions only have 8 of each value, so this must be constant
        if call_count == 9:
            return "constant", call_count

    #if we checked all inputs and they all matched, it is constant
    return "constant", call_count


#test 1: worst case for a constant function
#a constant function always returns the same value
#so we will keep seeing the same output until we hit 9 and stop
def always_true(a, b, c, d):
    return True

result, calls = determine_constant_balanced_optimal(always_true)
assert result == "constant"
assert calls == 9
print(f"constant worst case: {calls} calls (stopped after 9 identical outputs)")


#test 2: best case for a balanced function
#the parity function flips its output often, so we find a difference quickly
def parity(a, b, c, d):
    return (int(a) + int(b) + int(c) + int(d)) % 2 == 0

result, calls = determine_constant_balanced_optimal(parity)
assert result == "balanced"
print(f"balanced best case: {calls} calls (found a different output early)")


#test 3: worst case for a balanced function
#put all 8 True outputs first so the first False appears at position 9
all_inputs = list(product([False, True], repeat=4))
first_half = set(all_inputs[:8])

def balanced_true_first(a, b, c, d):
    return (a, b, c, d) in first_half

result, calls = determine_constant_balanced_optimal(balanced_true_first)
assert result == "balanced"
assert calls == 9
print(f"balanced worst case: {calls} calls (first different output appeared at call 9)")


#test 4: run the optimal algorithm on 1000 random functions
#and check that we never need more than 9 calls
max_calls = 0
for _ in range(1000):
    f = random_constant_balanced()
    _, n = determine_constant_balanced_optimal(f)
    if n > max_calls:
        max_calls = n

assert max_calls <= 9
print(f"\nover 1000 random functions, the most calls needed was: {max_calls}")
print("this confirms the maximum is 9 calls (= 2^(n-1) + 1 where n=4)")


constant worst case: 9 calls (stopped after 9 identical outputs)
balanced best case: 2 calls (found a different output early)
balanced worst case: 9 calls (first different output appeared at call 9)

over 1000 random functions, the most calls needed was: 9
this confirms the maximum is 9 calls (= 2^(n-1) + 1 where n=4)


# Problem 3: Quantum Oracles
Deutsch's algorithm is the simplest example of a quantum algorithm using superposition to determine a global property of a function with a single evaluation. In the single-input case, there are four possible Boolean functions. Using Qiskit, create the appropriate quantum oracles for each of the possible single-Boolean-input functions used in Deutsch's algorithm. Demonstrate their use and explain how each oracle implements its corresponding function.

# Problem 4: Deutsch's Algorithm with Qiskit
Use Qiskit to design a quantum circuit that solves Deutsch's problem for a function with a single Boolean input. Implement the necessary circuit and demonstrate its use with each of the quantum oracles from Problem 3. Describe how the interference pattern produced by the circuit allows you to determine whether the function is constant or balanced using only one query to the oracle.



# Problem 5: Scaling to the Deutsch–Jozsa Algorithm
The Deutsch–Jozsa algorithm generalizes Deutsch's approach to functions with several input bits. Use Qiskit to create a quantum circuit that can handle the four-bit functions generated in Problem 1. Explain how the classical function is encoded as a quantum oracle, and demonstrate the use of your circuit on both of the constant functions and any two balanced functions of your choosing. Show that the circuit correctly identifies the type of each function.